In [9]:
from pyarrow import dataset as ds
import pandas as pd
import numpy as np
import pickle
from collections import Counter

from pydtmc import MarkovChain, plot_graph
import matplotlib.pyplot as plt

### load dataset

In [10]:
dataset = ds.dataset("../data/renovation-data-all.parquet", format="parquet")
table = dataset.to_table(columns=None, use_threads=True)
df_preprocessed = table.to_pandas()
print(f"shape: {df_preprocessed.shape}")
df_preprocessed.head()

shape: (39736, 83)


,entity_id,epc_before,ontology.building.building_general.construction_year,ontology.building.building_general.floor_area,ontology.building.pv.available_roof_area_for_solar_panel_in_m2,ontology.building.pv.has_pv,steps,has_insulate_exterior_inclined_roof,has_insulate_exterior_flat_roof,has_insulate_exterior_wall,...,ontology.building.systems.space_heating_emission_systems.CONVECTOR,ontology.building.systems.space_heating_emission_systems.FLOOR,ontology.building.systems.space_heating_emission_systems.AIR,ontology.building.systems.space_heating_emission_systems.LOCAL_STOVE,ontology.building.envelope.insulation_level_roof_extension_MODERATELY_INSULATED,ontology.building.envelope.insulation_level_roof_main_MODERATELY_INSULATED,ontology.building.systems.space_heating_generation_on_electricity.CENTRAL_HEAT_PUMP,ontology.building.systems.space_heating_generation_on_electricity.LOCAL_RESISTANCE_HEATERS,ontology.building.envelope.basement_occurance_PARTIAL,ontology.building.envelope.floor_boundary_types.BASEMENT_OR_CRAWLSPACE
0,0,394,1965.0,203.0,8.0,0,"[insulate_floors_above_air_from_below, insulat...",1,1,1,...,1,0,0,0,0,0,0,0,0,1
1,1,413,1969.0,226.0,16.0,0,"[insulate_floors_above_air_from_below, insulat...",1,1,1,...,1,0,1,0,1,0,0,0,0,1
2,2,469,1967.0,195.0,71.0,0,"[insulate_exterior_inclined_roof, replace_sing...",1,1,1,...,0,0,0,0,1,0,0,0,0,1
3,3,592,1969.0,220.0,63.0,0,"[insulate_floors_above_air_from_below, replace...",0,1,0,...,1,1,0,0,0,0,0,0,0,1
4,5,511,1967.0,173.0,25.0,0,"[replace_epump_with_dhw, insulate_floors_above...",1,1,1,...,0,0,0,1,1,1,0,0,0,1


In [11]:
steps_encoding = {
    'insulate_exterior_inclined_roof': 1,
    'insulate_exterior_flat_roof': 2,
    'insulate_exterior_wall': 3,
    'insulate_cavity_wall': 4,
    'insulate_floors_above_air_from_below': 5,
    'replace_single_and_old_double_for_hr_glass_with_new_frames': 6,
    'replace_non_condensing_gas_boiler_with_condensing_gas_boiler': 7,
    'replace_epump_with_dhw': 8,
    'install_pv_panels': 9,  
}
code_to_step = {code: step for step, code in steps_encoding.items()}

class ThresholdedModel:
    def __init__(self, model, threshold: float = 0.5, positive_class_index: int = 1):
        self.model = model
        self.threshold = float(threshold)
        self.positive_class_index = int(positive_class_index)

    def fit(self, X, y):
        # Delegate to underlying sklearn model
        return self.model.fit(X, y)
    
    def predict_proba(self, X):
        # Delegate to underlying sklearn model
        return self.model.predict_proba(X)

    def decision_function(self, X):
        # Probability for the positive class
        proba = self.predict_proba(X)
        return proba[:, self.positive_class_index]

    def predict(self, X):
        scores = self.decision_function(X)
        return (scores >= self.threshold).astype(int)

def get_columns_to_drop(df, component):
    columns_to_drop = ["entity_id"]

    if component == "relevant_steps":
        columns_to_drop.extend([col for col in df.columns if col.startswith("has_")])
        columns_to_drop.extend(["step_count", "steps", "steps_gt"])
    elif component == "first_step":
        columns_to_drop.extend(["first_step", "steps", "steps_gt"]) # remove steps(gt) since list, but same information is in 'has_' columns

    return columns_to_drop

def plot_heatmap(M, found_steps):
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(M, cmap='Greens')

    labels = found_steps

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))

    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)

    plt.setp(ax.get_xticklabels(), rotation=0, ha="center", rotation_mode="anchor", fontsize=8)

    for i in range(len(labels)):
        for j in range(len(labels)):
            if M[i][j] > 0.35:
                color = "w"
            else:
                color = "black"
            ax.text(j, i, f"{M[i][j]:.2f}", ha="center", va="center", color=color)
    
    ax.set_ylabel("from state")
    ax.set_xlabel("to state")

    plt.show()

### load components

In [12]:
# component A (one-vs-all)
with open("../models/relevant_steps_RF.pkl", "rb") as f:
    relevant_steps = pickle.load(f)

relevant_steps_model = relevant_steps["models_RF"]

# component B
transition_matrix = np.load("../models/transition_matrix.npy")

# component C (multiclass)
with open("../models/first_step_RF.pkl", "rb") as f:
    first_step = pickle.load(f)

first_step_model = first_step["model_RF"]

## model

In [13]:
def write_to_logfile(S, N, scores, per_step, filename="../results/results.csv"):
    log_keeping = pd.DataFrame({'SIMULATIONS': S, 'SAMPLE_SIZE': N, **scores}, index=[0])
    for i in range(1, 10):
        log_keeping[f'step_{i}_accuracy'] = per_step["step_correct"][i-1] / per_step["step_total"][i-1] if per_step["step_total"][i-1] > 0 else np.nan
        log_keeping[f'step_{i}_occurrences'] = per_step["step_total"][i-1]

    log_keeping.to_csv(filename, mode='a', index=False, header=False)

def reduce_transition_matrix(trans_matrix, actions_to_take, normalize=True, visualize=False):
    reduced_transition_matrix = trans_matrix.copy()
    indices_to_remove = [steps_encoding[action]-1 for action in steps_encoding.keys() if action not in actions_to_take]
    reduced_transition_matrix = np.delete(reduced_transition_matrix, indices_to_remove, axis=0)
    reduced_transition_matrix = np.delete(reduced_transition_matrix, indices_to_remove, axis=1)

    if normalize:
        row_sums = reduced_transition_matrix.sum(axis=1, keepdims=True)
    
        # identify zero-sum rows to avoid division by zero
        zero_rows = (row_sums == 0)
    
        # only divide rows with non-zero sums.
        reduced_transition_matrix = np.divide(
            reduced_transition_matrix,
            row_sums,
            out=np.zeros_like(reduced_transition_matrix, dtype=float),
            where=~zero_rows
        )
    
        # rows that sum to zero: make absorbing
        if np.any(zero_rows):
            idx = np.flatnonzero(zero_rows.ravel())
            reduced_transition_matrix[idx, :] = 0.0
            reduced_transition_matrix[idx, idx] = 1.0

    if visualize:
        actions_to_take_encoded = [steps_encoding[action] for action in actions_to_take]
        plot_heatmap(reduced_transition_matrix, actions_to_take_encoded)

    return reduced_transition_matrix

def reset_scores():
    scores = {
        "correct actions": 0,
        "incorrect actions": 0,
        "correct first step": 0,
        "incorrect first step": 0,
        "identical": 0,
        "non-identical": 0,
        "non_unique_sequences": 0,
        "first_step_not_in_actions": 0,
        "fails": 0
    }

    per_step = {
        "step_correct": [0] * (len(steps_encoding) + 1),
        "step_total": [0] * (len(steps_encoding) + 1)
        }

    return scores, per_step

def update_scores(scores, per_step, ground_truth, final_prediction):
    # compare sets of actions (ignoring order) > COMPONENT A
    if set(ground_truth) == set(final_prediction):
        scores["correct actions"] += 1
    else:
        scores["incorrect actions"] += 1

    # compare first step > COMPONENT C
    if len(ground_truth) > 0:
        if len(final_prediction) > 0 and ground_truth[0] == final_prediction[0]:
            scores["correct first step"] += 1
        else:
            scores["incorrect first step"] += 1
    else:
        if ground_truth == final_prediction:
            scores["correct first step"] += 1
        else:
            scores["incorrect first step"] += 1

    # compare identical sequences (including order) > COMPONENT D
    if ground_truth == final_prediction:
        scores["identical"] += 1
    else:
        scores["non-identical"] += 1

    # keep track of per-step accuracy
    if len(ground_truth) == 0:
        per_step["step_total"][0] += 1
        if len(final_prediction) == 0:
            per_step["step_correct"][0] += 1
    else:
        for pos in range(len(ground_truth)):  # pos=0 -> index 1 in per_step
            idx = pos + 1
            per_step["step_total"][idx] += 1

            if pos < len(final_prediction) and final_prediction[pos] == ground_truth[pos]:
                per_step["step_correct"][idx] += 1

    return scores, per_step

In [14]:
# get predictions for component A (one-vs-all)
X_relevant_steps = df_preprocessed.drop(columns=get_columns_to_drop(df_preprocessed, "relevant_steps"))
pred_relevant_steps = pd.DataFrame(index=df_preprocessed.index)
pred_relevant_steps["entity_id"] = df_preprocessed["entity_id"]

for action_name, clf in relevant_steps_model.items():
    # probabilities p(class=1)
    proba = clf.predict_proba(X_relevant_steps)[:, 1]

    y_pred = (proba >= clf.threshold).astype(int)

    pred_relevant_steps[f"pred_{action_name}"] = y_pred
    pred_relevant_steps[f"proba_{action_name}"] = proba

# get predictions for component C (multiclass)
ctd_first_step = get_columns_to_drop(df_preprocessed, "first_step")
ctd_first_step.remove("first_step")
X_first_step = df_preprocessed.drop(columns=ctd_first_step)
y_class = first_step_model.predict(X_first_step)
proba_first_step = first_step_model.predict_proba(X_first_step)

pred_first_step = pd.DataFrame(index=df_preprocessed.index)
pred_first_step["entity_id"] = df_preprocessed["entity_id"]

pred_first_step["pred_first_step"] = y_class

for k in range(proba_first_step.shape[1]):
    pred_first_step[f"first_step_proba_{k}"] = proba_first_step[:, k]

In [15]:
S = 300 #number of simulations
N = df_preprocessed.shape[0] #number of samples to evaluate
VISUALIZE = False

scores, per_step = reset_scores()
samples = list(df_preprocessed.sample(N, random_state=42).index.values)

for run, sample in enumerate(samples):
    # find index of sample in df
    sample_index = df_preprocessed.index.get_loc(sample)

    # find ground truth steps for sample
    ground_truth = df_preprocessed.at[sample, "steps"]
    ground_truth_encoded = [steps_encoding[step] for step in ground_truth]

    # predict relevant renovation steps (component A)
    relevant_steps_row = pred_relevant_steps.loc[sample_index]
    relevant_steps_pred = [step.replace("pred_has_", "") for step in relevant_steps_row.index if step.startswith("pred_") and relevant_steps_row[step] == 1]
    relevant_steps_pred_encoded = [steps_encoding[step] for step in relevant_steps_pred]

    # get transition probabilities for predicted relevant steps (component B)
    reduced_transition_matrix = reduce_transition_matrix(transition_matrix, relevant_steps_pred, visualize=VISUALIZE, normalize=True)

    # predict first step
    first_step_pred_encoded = pred_first_step.at[sample, "pred_first_step"]
    first_step_pred = code_to_step.get(first_step_pred_encoded, None)

    # if no steps
    if len(relevant_steps_pred_encoded) == 0:
        final_prediction = []
        final_prediction_encoded = []
        
    # if only one step: final prediction = step
    elif len(relevant_steps_pred_encoded) == 1:
        final_prediction = relevant_steps_pred
        final_prediction_encoded = relevant_steps_pred_encoded
    
    # if two steps: final prediction = first + other
    elif len(relevant_steps_pred_encoded) == 2:
        if first_step_pred_encoded in relevant_steps_pred_encoded:
            final_prediction = [first_step_pred] + [step for step in relevant_steps_pred if step != first_step_pred]
            final_prediction_encoded = [first_step_pred_encoded] + [code for code in relevant_steps_pred_encoded if code != first_step_pred_encoded]
        else:
            final_prediction = relevant_steps_pred
            final_prediction_encoded = relevant_steps_pred_encoded

    # else: Markov Chain simulations (needs at least 3 steps)
    elif len(relevant_steps_pred_encoded) > 2:
        mc = MarkovChain(reduced_transition_matrix, [str(item) for item in relevant_steps_pred_encoded])

        if VISUALIZE:
            plot_graph(mc, dpi=100)

        if first_step_pred_encoded not in relevant_steps_pred_encoded:
            scores["first_step_not_in_actions"] += 1
            continue

        simulations = [
            mc.simulate(steps=len(relevant_steps_pred_encoded)-1, initial_state=str(first_step_pred_encoded))
            for _ in range(S)
        ]

        # keep only valid simulations: meaning each state is only visited once
        valid_simulations = [sim for sim in simulations if len(set(sim)) == len(sim)]

        sequence_count = Counter([tuple(subarray) for subarray in valid_simulations])

        if len(sequence_count) == 0:
            scores["fails"] += 1
            continue
        else:
            most_common_sequence, most_common_count = sequence_count.most_common(1)[0]
            max_elements = [element for element, count in sequence_count.items() if count == most_common_count]
    
            if len(max_elements) != 1:
                scores["non_unique_sequences"] += 1
    
            final_prediction = [code_to_step[int(code)] for code in most_common_sequence]
            final_prediction_encoded = [int(code) for code in most_common_sequence]

    scores, per_step = update_scores(scores, per_step, ground_truth_encoded, final_prediction_encoded)

write_to_logfile(S, N, scores, per_step)